# Arquitetura Híbrida PySpark → Pandas

## Tutorial de Uso para Datasets Massivos

Este notebook demonstra como usar a arquitetura híbrida implementada para processar datasets de Anti-Money Laundering que não cabem na memória.

## 🎯 Fluxo Completo

```
HI-Large_Trans.csv (5 GB) ──┐
                            │
HI-Large_accounts.csv ──────┤
                            ▼
                    ┌──────────────┐
                    │   PySpark    │ ← Amostragem validada
                    │   Sampler    │   (testes estatísticos)
                    └──────┬───────┘
                           │
                           ▼
              HI-Large_sampled.csv (200 MB)
                           │
                           ▼
                    ┌──────────────┐
                    │    Pandas    │ ← Split + Feature Eng.
                    │   Pipeline   │   + Modelagem
                    └──────┬───────┘
                           │
                           ▼
              df_treino.csv + df_oot.csv
```

## ✅ Passo 0: Validar Setup

In [ ]:
# Execute este script para verificar se tudo está instalado
!python validate_hybrid_setup.py

## 🚀 Passo 1: Executar Amostragem PySpark

**Execute APENAS UMA VEZ** (amostra fica salva):

In [ ]:
# Gerar amostra estatisticamente validada
# Tempo estimado: 5-15 minutos
!python source/spark_sampler.py

### O que o script faz:

1. **Carrega datasets com PySpark** (processamento distribuído)
2. **Join de transações com contas** (origem e destino)
3. **Calcula estatísticas da população**:
   - Médias de `Amount Received`, `Amount Paid`
   - Distribuições de `Payment Format`, `From Bank`, `To Bank`
   - Proporção de `Is Laundering`
4. **Gera amostra estratificada** (10% dos dados)
5. **Valida com testes estatísticos**:
   - Z-test para médias numéricas
   - Chi-quadrado para distribuições categóricas
6. **Repete até encontrar amostra válida** (p-value > 0.05)
7. **Salva em `data/interim/HI-Large_sampled.csv`**

## 📊 Passo 2: Verificar Amostra Gerada

In [ ]:
import pandas as pd
from pathlib import Path

# Carregar amostra gerada
sampled_path = Path('data/interim/HI-Large_sampled.csv')

if sampled_path.exists():
    df_sample = pd.read_csv(sampled_path)
    
    print(f"✓ Amostra carregada com sucesso!")
    print(f"  Shape: {df_sample.shape}")
    print(f"  Memória: {df_sample.memory_usage(deep=True).sum() / (1024**2):.2f} MB")
    print(f"\nColunas ({len(df_sample.columns)}):")
    print(df_sample.columns.tolist())
    print(f"\nDistribuição do Target:")
    print(df_sample['Is Laundering'].value_counts())
    
else:
    print("✗ Amostra não encontrada. Execute: python source/spark_sampler.py")

## 🔄 Passo 3: Processar com Pandas (Split + Feature Engineering)

Agora que temos a amostra, executamos o pipeline Pandas normalmente:

In [ ]:
# O dataset.py detecta automaticamente a amostra e processa ela
!python source/dataset.py

### O que acontece:

1. **Detecta `HI-Large_sampled.csv`** (se existir)
2. **Carrega com Pandas** (cabe na memória!)
3. **Executa split temporal/estratificado**:
   - `df_treino.csv` (80%)
   - `df_oot.csv` (20%)
4. **Salva em `data/processed/`**

## 📈 Passo 4: Verificar Dados Processados

In [ ]:
# Carregar dados de treino e OOT
treino_path = Path('data/processed/df_treino.csv')
oot_path = Path('data/processed/df_oot.csv')

if treino_path.exists() and oot_path.exists():
    df_treino = pd.read_csv(treino_path)
    df_oot = pd.read_csv(oot_path)
    
    print("✓ Dados de treino e OOT carregados!\n")
    
    print(f"Treino:")
    print(f"  Shape: {df_treino.shape}")
    print(f"  Target: {df_treino['Is Laundering'].value_counts().to_dict()}")
    
    print(f"\nOOT:")
    print(f"  Shape: {df_oot.shape}")
    print(f"  Target: {df_oot['Is Laundering'].value_counts().to_dict()}")
    
    # Primeiras linhas
    display(df_treino.head())
    
else:
    print("✗ Dados processados não encontrados. Execute: python source/dataset.py")

## 🎓 Justificativa Estatística

### Por que a amostra é representativa?

**Testes aplicados:**

#### 1. Z-Test (Variáveis Numéricas)
- **Hipótese Nula (H₀):** média_amostra = média_população
- **Estatística:** $Z = \frac{\bar{x}_{amostra} - \mu_{população}}{\sigma_{população} / \sqrt{n}}$
- **Critério:** p-value > 0.05 → Não rejeitamos H₀

#### 2. Chi-Quadrado (Variáveis Categóricas)
- **Hipótese Nula (H₀):** distribuição_amostra = distribuição_população
- **Estatística:** $\chi^2 = \sum \frac{(O_i - E_i)^2}{E_i}$
- **Critério:** p-value > 0.05 → Não rejeitamos H₀

**Conclusão:** Se todos os testes passam (p > 0.05), a amostra é estatisticamente indistinguível da população!

## ⚙️ Configurações Avançadas

### Ajustar tamanho da amostra

Edite `source/spark_sampler.py`:

```python
SAMPLE_FRACTION = 0.15  # 15% ao invés de 10%
```

### Ajustar rigor estatístico

```python
P_VALUE_THRESHOLD = 0.01  # Mais rigoroso (1% ao invés de 5%)
```

### Ajustar tentativas

```python
MAX_ITERATIONS = 20  # Mais tentativas
```

## 🔍 Diagnóstico de Problemas

### Problema: "Nenhuma amostra válida encontrada"

**Soluções:**
1. Aumente `SAMPLE_FRACTION` (de 0.10 para 0.15 ou 0.20)
2. Aumente `MAX_ITERATIONS` (de 10 para 20)
3. Relaxe `P_VALUE_THRESHOLD` (de 0.05 para 0.10)

### Problema: "OutOfMemoryError no Spark"

**Solução:** Aumente memória do driver:
```python
spark = SparkSession.builder \
    .config("spark.driver.memory", "12g") \
    .getOrCreate()
```

## 📚 Próximos Passos

Após ter `df_treino.csv` e `df_oot.csv`:

1. **Feature Engineering** → `05_feature_engineering.ipynb`
2. **Treinamento** → `08_treinamento_timeseriesplit.ipynb`
3. **Avaliação** → Comparar performance em treino vs. OOT

---

**Sucesso! 🚀** Você agora tem uma arquitetura escalável para processar datasets massivos!